# Deney Düzenleme | 26 Kasım 2025

Latin Kare Düzeni

In [18]:
veri <- data.frame(
    satir = rep(c("Açık", "Orta Açık", "Orta", "Koyu"), each = 4),
    sutun = rep(c("85C", "90C", "95C", "100C"), times = 4),
    islem = c("A", "C", "B", "D",
              "D", "B", "A", "C",
              "C", "A", "D", "B",
              "B", "D", "C", "A"),
    lezzet = c(60,88,66,68, # Açık
               76,78,72,96, # Orta Açık
               92,86,90,74, # Orta
               68,86,91,66) # Koyu
)
View(veri)

satir,sutun,islem,lezzet
<chr>,<chr>,<chr>,<dbl>
Açık,85C,A,60
Açık,90C,C,88
Açık,95C,B,66
Açık,100C,D,68
Orta Açık,85C,D,76
Orta Açık,90C,B,78
Orta Açık,95C,A,72
Orta Açık,100C,C,96
Orta,85C,C,92


In [19]:
veri$islem_adi <- factor(
    veri$islem,
    levels = c("A", "B", "C", "D"),
    labels = c("Chemex", "French Press", "V60", "Aeropress")
)
veri$satir <- factor(veri$satir, levels = c("Açık", "Orta Açık", "Orta", "Koyu"))
veri$sutun <- factor(veri$sutun)
veri$islem <- factor(veri$islem)

View(veri)
summary(veri$lezzet)

satir,sutun,islem,lezzet,islem_adi
<fct>,<fct>,<fct>,<dbl>,<fct>
Açık,85C,A,60,Chemex
Açık,90C,C,88,V60
Açık,95C,B,66,French Press
Açık,100C,D,68,Aeropress
Orta Açık,85C,D,76,Aeropress
Orta Açık,90C,B,78,French Press
Orta Açık,95C,A,72,Chemex
Orta Açık,100C,C,96,V60
Orta,85C,C,92,V60


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  60.00   68.00   77.00   78.56   88.50   96.00 

In [20]:
library(dplyr)
cat("--- Genel Değerlendirme ---\n")
# işlemlere göre ortalamalar
islem_ozet <- veri %>% group_by(islem_adi) %>% summarise(ortalama=mean(lezzet), sd = sd(lezzet), n=n())
print(islem_ozet)

# satırlara göre özet
# kavrulma derecesine göre ortalamalar
satir_ozet <- veri %>% group_by(satir) %>% summarise(ortalama=mean(lezzet), sd = sd(lezzet), n=n())
print(satir_ozet)

# sutunlara göre özet
# su sıcaklığına göre ortalamalar
sutun_ozet <- veri %>% group_by(sutun) %>% summarise(ortalama=mean(lezzet), sd = sd(lezzet), n=n())
print(sutun_ozet)

--- Genel Değerlendirme ---
# A tibble: 4 × 4
  islem_adi    ortalama    sd     n
  <fct>           <dbl> <dbl> <int>
1 Chemex           71   11.1      4
2 French Press     71.5  5.51     4
3 V60              91.8  3.30     4
4 Aeropress        80    9.93     4
# A tibble: 4 × 4
  satir     ortalama    sd     n
  <fct>        <dbl> <dbl> <int>
1 Açık          70.5 12.2      4
2 Orta Açık     80.5 10.6      4
3 Orta          85.5  8.06     4
4 Koyu          77.8 12.6      4
# A tibble: 4 × 4
  sutun ortalama    sd     n
  <fct>    <dbl> <dbl> <int>
1 100C      76   13.8      4
2 85C       74   13.7      4
3 90C       84.5  4.43     4
4 95C       79.8 12.7      4


### Anova

In [21]:
model <- aov(lezzet ~ islem_adi + satir + sutun, data = veri)
anova_sonuc <- summary(model)
print(anova_sonuc)

            Df Sum Sq Mean Sq F value   Pr(>F)    
islem_adi    3 1132.2   377.4  34.637 0.000349 ***
satir        3  470.2   156.7  14.384 0.003792 ** 
sutun        3  256.2    85.4   7.837 0.016918 *  
Residuals    6   65.4    10.9                     
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


```R
            Df Sum Sq Mean Sq F value   Pr(>F)    
islem_adi    3 1132.2   377.4  34.637 0.000349 ***
satir        3  470.2   156.7  14.384 0.003792 ** 
sutun        3  256.2    85.4   7.837 0.016918 *  
Residuals    6   65.4    10.9                       
```

Hipotezler:

- H0: Kahve demleme yöntemlerinin lezzet puan ortalamaları arasında fark yoktur.
- H1: En az bir kahve demleme yönteminin lezzet puan ortalaması diğer yöntemlerin lezzet puan ortalamalarından farklıdır.

P değeri 0.003792 < 0.05 olduğundan, H0 reddedilir. En az bir yöntemin lezzet puan ortalaması diğerlerinden farklıdır.

Hangi yöntemlerin farklı olduğunu belirlemek için Post-Hoc Tukey HSD testi uygulanır.

In [22]:
tukey_islem <- TukeyHSD(model, "islem_adi")
print(tukey_islem)

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = lezzet ~ islem_adi + satir + sutun, data = veri)

$islem_adi
                         diff         lwr       upr     p adj
French Press-Chemex      0.50  -7.5799018  8.579902 0.9961482
V60-Chemex              20.75  12.6700982 28.829902 0.0004696
Aeropress-Chemex         9.00   0.9200982 17.079902 0.0320010
V60-French Press        20.25  12.1700982 28.329902 0.0005380
Aeropress-French Press   8.50   0.4200982 16.579902 0.0406946
Aeropress-V60          -11.75 -19.8299018 -3.670098 0.0094273



In [23]:
tukey_satir <- TukeyHSD(model, "satir")
print(tukey_satir)

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = lezzet ~ islem_adi + satir + sutun, data = veri)

$satir
                diff         lwr        upr     p adj
Orta Açık-Açık 10.00   1.9200982 18.0799018 0.0201181
Orta-Açık      15.00   6.9200982 23.0799018 0.0027359
Koyu-Açık       7.25  -0.8299018 15.3299018 0.0758225
Orta-Orta Açık  5.00  -3.0799018 13.0799018 0.2412510
Koyu-Orta Açık -2.75 -10.8299018  5.3299018 0.6604562
Koyu-Orta      -7.75 -15.8299018  0.3299018 0.0589165



```R
Orta Açık-Açık 10.00   1.9200982 18.0799018 0.0201181
Orta-Açık      15.00   6.9200982 23.0799018 0.0027359
```

In [24]:
tukey_sutun <- TukeyHSD(model, "sutun")
print(tukey_sutun)

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = lezzet ~ islem_adi + satir + sutun, data = veri)

$sutun
          diff         lwr       upr     p adj
85C-100C -2.00 -10.0799018  6.079902 0.8263836
90C-100C  8.50   0.4200982 16.579902 0.0406946
95C-100C  3.75  -4.3299018 11.829902 0.4409843
90C-85C  10.50   2.4200982 18.579902 0.0160872
95C-85C   5.75  -2.3299018 13.829902 0.1642169
95C-90C  -4.75 -12.8299018  3.329902 0.2735829



In [ ]:
library(haven)
write_sav(veri, "latin_kare_duzeni.sav")
# "~/Downloads/latin_kare_duzeni.sav"